# Support Vector Machines and ADMM

Direct SVM and a consensus ADMM implementation on partitioned data.

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

np.random.seed(1)
n_samples, n_features = 80, 2
X_pos = np.random.randn(n_samples // 2, n_features) + 1.5
X_neg = np.random.randn(n_samples // 2, n_features) - 1.5
X_all = np.vstack([X_pos, X_neg])
y_all = np.hstack([np.ones(n_samples // 2), -np.ones(n_samples // 2)])
lambda_param = 0.1

a = cp.Variable(n_features)
b = cp.Variable()
hinge = cp.sum(cp.pos(1 - cp.multiply(y_all, X_all @ a + b)))
prob = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(a) + lambda_param * hinge))
prob.solve()
print('Direct SVM a, b, obj:', a.value, b.value, prob.value)

m = 4
X = np.split(X_all, m)
y = np.split(y_all, m)
rho, max_iters, tol = 0.2, 50, 1e-4
a_loc = [cp.Variable(n_features) for _ in range(m)]
b_loc = [cp.Variable() for _ in range(m)]
z_a = np.zeros(n_features)
z_b = 0.0
u_a = [np.zeros(n_features) for _ in range(m)]
u_b = [0.0 for _ in range(m)]
for iteration in range(max_iters):
    for i in range(m):
        hinge_loss = cp.sum(cp.pos(1 - cp.multiply(y[i], X[i] @ a_loc[i] + b_loc[i])))
        local_obj = (lambda_param / (2 * m)) * cp.sum_squares(a_loc[i]) + hinge_loss \
            + u_a[i] @ (a_loc[i] - z_a) + u_b[i] * (b_loc[i] - z_b) \
            + (rho / 2) * cp.sum_squares(a_loc[i] - z_a) + (rho / 2) * cp.square(b_loc[i] - z_b)
        cp.Problem(cp.Minimize(local_obj)).solve()
    z_a = np.mean([a_loc[i].value + u_a[i] / rho for i in range(m)], axis=0)
    z_b = np.mean([b_loc[i].value + u_b[i] / rho for i in range(m)])
    for i in range(m):
        u_a[i] += rho * (a_loc[i].value - z_a)
        u_b[i] += rho * (b_loc[i].value - z_b)
print('ADMM z_a, z_b:', z_a, z_b)
